# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven3"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    # Not a plain `git pull` -- if this clone has ANY local changes (e.g. leftover
    # checkpoints/outputs from an earlier run in the same runtime that never got pushed),
    # a pull can fail outright ("local changes would be overwritten") and Colab just
    # prints the error and moves on -- training then silently proceeds on stale code with
    # no visible failure until much later (e.g. a non-fast-forward push at the end).
    # fetch + hard reset guarantees this checkout exactly matches origin/{BRANCH} no
    # matter what state it was left in.
    !cd ECE1508_GenAI && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

%cd ECE1508_GenAI
!git log --oneline -1


remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 10 (delta 8), reused 10 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 5.10 KiB | 870.00 KiB/s, done.
From https://github.com/WoodyChang21/ECE1508_GenAI
 * branch            steven3    -> FETCH_HEAD
   50515af..638777d  steven3    -> origin/steven3
HEAD is now at 638777d Fix colab_train.ipynb: clone/push against steven3, not stale steven2
/content/ECE1508_GenAI
638777d (HEAD -> steven2, origin/steven3) Fix colab_train.ipynb: clone/push against steven3, not stale steven2


In [3]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [5]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 81 items                                                             

steven/tests/test_cvae_inpainting.py::test_default_decoder_ctx_dim_matches_old_behavior PASSED [  1%]
steven/tests/test_cvae_inpainting.py::test_decoder_ctx_dim_bottlenecks_decoder_input PASSED [  2%]
steven/tests/test_cvae_inpainting.py::test_prior_head_still_sees_full_ctx_dim_when_bottlenecked PASSED [  3%]
steven/tests/test_cvae_inpainting.py::test_decoder_output_width_doubled_for_mean_and_logvar PASSED [  4%]
steven/tests/test_cvae_inpainting.py::test_price_logvar_stays_within_configured_range PASSED [  6%]
steven/tests/test_cvae_inpainting.py::test_vol_logvar_stays_within_configured_range PASSED [  7%]
steven/tests/test_cvae_inpain

## Momentum feature setup (EMA9/EMA21 + RSI-14 + VIX)

Both models below train on the base hourly OHLCV plus three added features (`src/momentum_pipeline.py`)
-- EMA9/EMA21 crossover, RSI-14, and VIX (previous trading day's close, a genuine market-derived
sentiment signal, not a transform of SPY's own price). VIX isn't in the hourly parquet, so it's pulled
fresh here via `yfinance` before training. Fresh pull each run rather than committing the parquet --
cheap (a few seconds), avoids the data going stale, matches how the rest of this notebook already
re-clones/re-installs fresh each time.

In [6]:
!pip install -q -r steven/requirements-probe.txt
!python steven/src/collect_vix_yfinance.py

22:35:03 wrote 8139 VIX daily bars (1993-01-29 to 2025-05-29) to /content/ECE1508_GenAI/steven/data/vix_daily_yfinance.parquet


## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst_hourly_momentum.yaml)

Same architecture/training recipe as `configs/patchtst.yaml`, plus the momentum features from the setup
cell above (`PatchTST` gained an `n_feature_channels` param for this -- see `src/models/patchtst.py`).
Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first
instead of the full config.

In [7]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst_hourly_momentum.yaml --device auto

22:35:05 device: cuda
22:35:05 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:35:05 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:35:05 momentum features enabled: ema_cross/trend_position/rsi/vix, N_FEATURE_CHANNELS=11
22:35:09 epoch 1/20  train_loss=0.23837  val_loss=0.13126  (2.4s)
22:35:09   -> saved best checkpoint (val_loss=0.13126) to steven/outputs/patchtst_checkpoint.pt
22:35:11 epoch 2/20  train_loss=0.16639  val_loss=0.11737  (2.0s)
22:35:11   -> saved best checkpoint (val_loss=0.11737) to steven/outputs/patchtst_checkpoint.pt
22:35:13 epoch 3/20  train_loss=0.15368  val_loss=0.10626  (2.0s)
22:35:13   -> saved best checkpoint (val_loss=0.10626) to steven/outputs/patchtst_checkpoint.pt
22:35:15 epoch 4/20  train_loss=0.14402  val_loss=0.11179  (2.1s)
22:35:17 epoch 5/20  train_loss=0.13872  val_loss=0.10042  (2.0s)
22:35:17   -> saved best checkpoint (val_loss=0.10042) to steven/outputs/patchtst_checkpoint.pt
2

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae_hourly_momentum.yaml)

Same architecture/loss recipe as `configs/cvae.yaml` (z_dim=8, decoder_ctx_dim=8, price_scale,
w_direction -- unchanged, extensively tested and never found to be the bottleneck), plus the momentum
features from the setup cell above.

In [8]:
!python steven/src/train_cvae.py --config steven/configs/cvae_hourly_momentum.yaml --device auto

22:35:51 device: cuda
22:35:51 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:35:51 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:35:51 momentum features enabled: ema_cross/trend_position/rsi/vix, N_CHANNELS=13
22:35:51 price_scale (open_ret, body_ret, upper_wick, lower_wick): [0.002682909369468689, 0.0028937843162566423, 0.0016679799882695079, 0.0017456382047384977]
22:35:55 epoch 1/30  beta=0.20  recon_mode=mse  train_recon=402.81513 (kl=24.9260 dir=2.7424 price_std=0.00720 vol_std=0.660)  val_recon=18.35774 (kl=22.2809 dir=2.6064 price_std=0.00783 vol_std=1.037)  gen_diversity_ratio=2.892 gen_crps=0.004358 gen_var_ratio=9.032  (2.3s)
22:35:55   -> saved best checkpoint (val_recon=18.35774) to steven/outputs/cvae_checkpoint.pt
22:35:56 epoch 2/30  beta=0.40  recon_mode=mse  train_recon=10.04180 (kl=10.6048 dir=1.5994 price_std=0.00722 vol_std=0.860)  val_recon=4.67873 (kl=4.4052 dir=0.9915 price_std=0.00671 vol_std=0.6

## Daily CVAE (EMA9/EMA21 + RSI-14 + VIX, rolling-window training) -- consolidated build

Replaces the earlier context-length-sweep probe (`probe_daily_cvae.py`, obsolete -- superseded
by everything below). This is the current best-tested daily-bars recipe, consolidated into one
build after extensive investigation (see `cvae_direction_collapse.md`'s "momentum enrichment",
"Adding VIX", and "robustness check" sections for the full history):

- **Data**: SPY's full 1993-2025 daily history pulled fresh via `yfinance`
  (`src/collect_daily_yfinance.py`) -- not a resample of the hourly parquet, which would only
  reach back to 2010. VIX pulled the same way (`src/collect_vix_yfinance.py`) as a genuine
  market-derived sentiment feature, not a transform of SPY's own price.
- **Features**: the same 4 price/wick components + volume as the hourly project, plus
  EMA9/EMA21 crossover, RSI-14, and VIX (previous trading day's close, so it's genuinely
  known before that day's open -- see `src/momentum_pipeline.py`).
- **Context**: 10 trading days -- matched calendar lookback vs. the hourly model's 70 bars
  (not bar count), sampled with a rolling window (every valid window used exactly once per
  epoch, not `WindowSampler`'s random multi-length draw -- see `probe_momentum_rolling_cvae.py`'s
  module docstring for why).
- **Architecture/loss**: unchanged from `configs/cvae.yaml`'s hourly recipe (z_dim=8,
  decoder_ctx_dim=8, price_scale, w_direction) -- extensively tested and never found to be the
  bottleneck, so kept constant here for comparability rather than re-tuned.
- **Trading strategy**: the same bracket take-profit/stop-loss walk-forward as the hourly
  project, but `stop_loss_pct` and `min_return_threshold` are both re-derived from daily's own
  measured volatility (`sell_bound`, the p99 3-day anchored move) instead of reusing the two
  hourly-tuned constants -- printed in this run's own log/report below.

**Headline finding going in, so the result below isn't a surprise**: a robustness check across
5 training seeds found CVAE's correlation with real daily market direction is statistically
indistinguishable from zero (mean +0.09, std 0.16 -- individual runs have landed anywhere from
-0.17 to +0.29 purely from training randomness). This build won't manufacture a real edge that
isn't there; it's still worth training and inspecting directly.

Writes to `cvae_checkpoint_momentum_rolling_daily.pt` -- does not touch or overwrite the hourly
`cvae_checkpoint.pt` above.

In [9]:
# yfinance isn't needed by the main hourly pipeline -- kept in its own requirements file
# (steven/requirements-probe.txt) rather than requirements-model.txt.

# !pip install -q -r steven/requirements-probe.txt

# Fresh pull each run rather than committing the parquets -- cheap (a few seconds), avoids
# the data going stale relative to whatever TEST_END this project is using, and matches how
# the rest of this notebook already re-clones/re-installs fresh each time.

# !python steven/src/collect_daily_yfinance.py
# !python steven/src/collect_vix_yfinance.py

In [10]:
# import sys
# sys.path.insert(0, "steven")
# import probe_momentum_rolling_cvae as pmr
# from IPython.display import Markdown, display

# results = pmr.main("daily")
# display(Markdown(pmr.format_report(results)))

## Compare CVAE vs. PatchTST across market scenarios (charts only)

Purely visual, no numbers: classifies test windows into uptrend/downtrend/choppy context (causally, from a trend z-score over the last 20 context bars -- see `classify_trend`), picks one representative window per label, and renders it as its own 3-panel image -- ground truth on the left, PatchTST's single predicted path top right, one CVAE sampled draw bottom right. Just candlesticks + a red box marking the generated region -- no buy/sell lines, no trade decisions, no metrics.json. `src/evaluate.py`'s old random-sample backtest (win rate, take-profit rate, etc.) is still defined in the file but no longer called by `main()` (see the module's git history if you want it back). Momentum features are auto-detected from each checkpoint's own saved config. Writes `uptrend.png`/`downtrend.png`/`choppy.png` to `steven/outputs/scenario_charts/`.

In [11]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

22:36:46 device: cuda
22:36:47 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:36:47 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:36:47 momentum features enabled: ema_cross/trend_position/rsi/vix, N_CHANNELS=13
22:36:47 rendering a trend scenario comparison (ctx_bars=70, k_draws=5) -- purely visual, no metrics/backtest/trade decisions
22:36:50 wrote trend comparison chart to steven/outputs/scenario_charts/trend_comparison.png


## Refresh v1.md from this run (disabled -- needs the old backtest metrics)

**Skip this unless you've restored the old backtest call in `evaluate.py`'s `main()`.** `update_report.py` hard-depends on `steven/outputs/metrics.json` + `steven/outputs/sample_plots/samples.json`, which the chart-only `evaluate.py` above no longer writes -- running it now either fails outright or silently rewrites `v1.md` from a stale metrics.json left over from an old run. Left commented out below rather than removed.

In [12]:
# !python steven/src/update_report.py

## Generative-quality CVAE: NLL + learned variance (configs/cvae_generative.yaml)

This project's actual goal is generating diverse, plausible, context-appropriate future candles -- not trading (see `cvae_direction_collapse.md`'s "generative pivot" discussion). The cells above train/evaluate the trading-framed checkpoints and stay as-is; this section is additive, not a replacement.

`configs/cvae_generative.yaml` is the same architecture as `configs/cvae_hourly_momentum.yaml` (momentum features on, `z_dim=8`/`ctx_dropout=0.3`/`decoder_ctx_dim=8`), but the decoder now predicts a per-component variance and trains against a real NLL (Laplace on open_ret/body_ret, Gaussian on wicks/volume) instead of plain MSE, with a 5-epoch mean-only warmup to avoid the classic "inflate variance instead of improving the mean" pathology. `w_direction` is disabled (0.0) -- it never beat chance at its own trading-motivated goal and has no generative-quality justification.

In [13]:
!python steven/src/train_cvae.py --config steven/configs/cvae_generative.yaml --device auto

22:36:53 device: cuda
22:36:53 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:36:53 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:36:53 momentum features enabled: ema_cross/trend_position/rsi/vix, N_CHANNELS=13
22:36:53 price_scale: disabled (reconstruction=nll -- learned variance replaces it)
22:36:57 epoch 1/30  beta=0.20  recon_mode=mse  train_recon=0.47182 (kl=1.2018 dir=0.0000 price_std=0.00688 vol_std=0.480)  val_recon=0.24863 (kl=1.2000 dir=0.0000 price_std=0.00681 vol_std=0.481)  gen_diversity_ratio=2.090 gen_crps=0.003781 gen_var_ratio=4.674  (2.3s)
22:36:57   -> saved best checkpoint (val_recon=0.24863) to steven/outputs/cvae_checkpoint_generative.pt
22:36:59 epoch 2/30  beta=0.40  recon_mode=mse  train_recon=0.31615 (kl=1.2006 dir=0.0000 price_std=0.00677 vol_std=0.478)  val_recon=0.19664 (kl=1.2000 dir=0.0000 price_std=0.00675 vol_std=0.481)  gen_diversity_ratio=1.695 gen_crps=0.003024 gen_var_ratio=3.119  (1

## Evaluate generative quality: diversity, calibration, context-sensitivity

Runs `src/evaluate_generative.py` over a full deterministic rolling-window pass (default `ctx_bars=70`, `k=32` samples/window) and reports, in model-native (log-return/z-scored) units:
- **Diversity** -- are a window's k sampled draws meaningfully different from each other?
- **Calibration** -- CRPS + rank histogram from the k samples directly, plus PIT/coverage curves using the learned variance (only for a `reconstruction: nll` checkpoint).
- **Context-sensitivity** -- the property this pivot is actually about: does generated output shift across realized-volatility regimes the way real data does (`effect_ratio` ≈ 1 is good, ≈ 0 means context-blind generation), and does conditioning on context beat a context-blind climatology baseline at all (`crps_skill_score`)?

Writes `steven/outputs/generative_metrics.json` and two plots to `steven/outputs/generative_plots/`: a regime x k-samples grid (rows = low/mid/high volatility buckets, columns = ground truth + generated draws) and a diversity fan chart (k sampled close-price paths overlaid on one context window).

In [14]:
!python steven/src/evaluate_generative.py \
  --cvae-checkpoint steven/outputs/cvae_checkpoint_generative.pt \
  --device auto

22:37:50 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:37:50 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:37:50 momentum features enabled: ema_cross/trend_position/rsi/vix, N_CHANNELS=13
22:37:50 evaluating on 2397 windows (ctx_bars=70, split=test, k=32, reconstruction=nll)
22:37:51 wrote metrics to steven/outputs/generative_metrics.json
22:37:54 wrote regime grid to steven/outputs/generative_plots/regime_grid.png
22:37:54 wrote diversity fan to steven/outputs/generative_plots/diversity_fan.png


## Sync results back to GitHub

Commits `steven/outputs/` (checkpoints, sample charts) from this Colab runtime and pushes straight to the `steven3` branch -- no manual zip/download step. That step wasn't reliably reaching the local machine: `files.download()`'s browser-download trick only works from the Colab web UI, not when this kernel is attached remotely (e.g. from VS Code's kernel picker), so nothing ever landed on disk. `steven/v1.md` isn't part of this commit -- the "Refresh v1.md" step above is disabled (see its own note).

Needs a GitHub personal access token with `repo` write scope for this push only -- entered via `getpass` below, never written to the notebook or committed anywhere.

In [15]:
# %%bash
# git fetch origin steven3
# git merge origin/steven3 --no-edit

In [16]:
import getpass

token = getpass.getpass("GitHub PAT (repo write, used only for this push): ")

In [17]:
%%bash -s "$token"
TOKEN="$1"
if [ -z "$TOKEN" ]; then
  echo "Token was empty -- re-run the getpass cell above and actually paste your PAT before pressing Enter." >&2
  exit 1
fi
git config user.email "colab@ephemeral.local"
git config user.name "Colab Runtime"
git add steven/outputs
if git diff --cached --quiet; then
  echo "Nothing new to commit -- outputs unchanged from last commit."
else
  git commit -m "Retrain + refresh chart comparison from Colab run"
fi
# Push unconditionally -- a prior run may have committed but failed to push (e.g. a blank
# token), in which case there's nothing new to commit here but HEAD is still ahead of origin.
# Pushes to steven3, not steven/steven2 -- this notebook and this branch are the working copy for
# now; steven is left alone so a second person's in-flight work there can't collide with
# this runtime's pushes.
git push "https://${TOKEN}@github.com/WoodyChang21/ECE1508_GenAI.git" HEAD:steven3

[steven2 011a1a5] Retrain + refresh chart comparison from Colab run
 7 files changed, 240 insertions(+)
 rewrite steven/outputs/cvae_checkpoint.pt (92%)
 rewrite steven/outputs/cvae_checkpoint_generative.pt (93%)
 create mode 100644 steven/outputs/cvae_checkpoint_pre_fix_repro.pt
 create mode 100644 steven/outputs/generative_metrics.json
 create mode 100644 steven/outputs/generative_plots/diversity_fan.png
 create mode 100644 steven/outputs/generative_plots/regime_grid.png
 create mode 100644 steven/outputs/scenario_charts/trend_comparison.png


To https://github.com/WoodyChang21/ECE1508_GenAI.git
   638777d..011a1a5  HEAD -> steven3


### Fallback: zip + browser download

Only useful if you're running this notebook inside the actual Colab web UI (not a remote kernel) and would rather download a zip than push through git.

In [18]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

updating: steven/outputs/ (stored 0%)
updating: steven/outputs/metrics.json (deflated 69%)
updating: steven/outputs/cvae_checkpoint.pt (deflated 6%)
updating: steven/outputs/sample_plots/ (stored 0%)
updating: steven/outputs/sample_plots/samples.json (stored 0%)
updating: steven/outputs/patchtst_checkpoint.pt (deflated 9%)
  adding: steven/outputs/generative_plots_nll/ (stored 0%)
  adding: steven/outputs/generative_plots_nll/regime_grid.png (deflated 8%)
  adding: steven/outputs/generative_plots_nll/diversity_fan.png (deflated 5%)
  adding: steven/outputs/generative_metrics_nll.json (deflated 73%)
  adding: steven/outputs/generative_metrics.json (deflated 72%)
  adding: steven/outputs/scenario_charts/ (stored 0%)
  adding: steven/outputs/scenario_charts/trend_comparison.png (deflated 15%)
  adding: steven/outputs/cvae_checkpoint_generative.pt (deflated 7%)
  adding: steven/outputs/generative_metrics_mse_baseline.json (deflated 73%)
  adding: steven/outputs/cvae_checkpoint_generative_m

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: reproduce the pre-fix CVAE checkpoint (comparison experiment)

Commit `2c4ad99` (before `price_scale`, `decoder_ctx_dim`, or `w_direction` existed) is the
best walk-forward result CVAE has produced (+6.79% total return, 66.4% trade rate). Every fix
attempted since has made the loss more "correct" by the direction-collapse diagnosis but
hasn't matched it. `configs/cvae_pre_fix_repro.yaml` reproduces that exact config (writes to
a separate checkpoint so it doesn't clobber `configs/cvae.yaml`'s), and
`src/diagnose_cvae_direction.py` reruns the same variance/correlation/eligibility checks used
throughout `cvae_direction_collapse.md` against any checkpoint -- run it against both to see
whether the fixes actually helped or whether `2c4ad99` was a favorable roll against one test
path. See `cvae_direction_collapse.md`'s "Revisiting the pre-collapse-chasing checkpoint".

In [19]:
!python steven/src/train_cvae.py --config steven/configs/cvae_pre_fix_repro.yaml --device auto

22:38:06 device: cuda
22:38:06 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:38:06 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:38:06 price_scale: disabled (use_price_scale=false)
22:38:10 epoch 1/30  beta=0.20  recon_mode=mse  train_recon=0.43496 (kl=1.2015 dir=0.0000 price_std=0.00694 vol_std=0.486)  val_recon=0.19605 (kl=1.2000 dir=0.0000 price_std=0.00688 vol_std=0.501)  gen_diversity_ratio=1.559 gen_crps=0.003064 gen_var_ratio=2.598  (2.2s)
22:38:10   -> saved best checkpoint (val_recon=0.19605) to steven/outputs/cvae_checkpoint_pre_fix_repro.pt
22:38:11 epoch 2/30  beta=0.40  recon_mode=mse  train_recon=0.30334 (kl=1.2000 dir=0.0000 price_std=0.00693 vol_std=0.479)  val_recon=0.18895 (kl=1.2000 dir=0.0000 price_std=0.00682 vol_std=0.487)  gen_diversity_ratio=1.538 gen_crps=0.002877 gen_var_ratio=2.583  (1.6s)
22:38:11   -> saved best checkpoint (val_recon=0.18895) to steven/outputs/cvae_checkpoint_pre_fix_repro.p

In [20]:
print("=== current checkpoint (price_scale + decoder_ctx_dim + w_direction) ===")
!python steven/src/diagnose_cvae_direction.py --cvae-checkpoint steven/outputs/cvae_checkpoint.pt

print("\n=== pre-fix repro checkpoint (matches commit 2c4ad99) ===")
!python steven/src/diagnose_cvae_direction.py --cvae-checkpoint steven/outputs/cvae_checkpoint_pre_fix_repro.pt

=== current checkpoint (price_scale + decoder_ctx_dim + w_direction) ===
checkpoint: steven/outputs/cvae_checkpoint.pt
model config: {'hidden': 32, 'ctx_dim': 64, 'z_dim': 8, 'decoder_hidden': 128, 'ctx_dropout': 0.3, 'decoder_ctx_dim': 8}
loss config: {'w_price': 1.0, 'w_vol': 0.5, 'use_price_scale': True, 'kl_cycles': 3, 'kl_ramp_fraction': 0.5, 'free_bits': 0.15, 'w_direction': 1.0, 'direction_temperature': 0.003}

Traceback (most recent call last):
  File "/content/ECE1508_GenAI/steven/src/diagnose_cvae_direction.py", line 170, in <module>
    main()
  File "/content/ECE1508_GenAI/steven/src/diagnose_cvae_direction.py", line 113, in main
    cvae.load_state_dict(ckpt["model_state"])
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 2639, in load_state_dict
    raise RuntimeError(
RuntimeError: Error(s) in loading state_dict for CVAEInpainting:
	size mismatch for context_encoder.net.0.weight: copying a param with shape torch.Size([32, 13, 5]) from che